In [ ]:
!unzip Resized.zip

Streaming output truncated to the last 5000 lines.
  inflating: Resized/Benign/ISIC_4920148_640x360.jpg  
  inflating: Resized/Benign/ISIC_4920166_640x360.jpg  
  inflating: Resized/Benign/ISIC_4924690_640x360.jpg  
  inflating: Resized/Benign/ISIC_4925442_640x480.jpg  
  inflating: Resized/Benign/ISIC_4932000_640x360.jpg  
  inflating: Resized/Benign/ISIC_4933150_640x360.jpg  
  inflating: Resized/Benign/ISIC_4933735_480x480.jpg  
  inflating: Resized/Benign/ISIC_4938188_640x360.jpg  
  inflating: Resized/Benign/ISIC_4938388_640x360.jpg  
  inflating: Resized/Benign/ISIC_4938638_640x480.jpg  
  inflating: Resized/Benign/ISIC_4939321_640x480.jpg  
  inflating: Resized/Benign/ISIC_4940947_640x427.jpg  
  inflating: Resized/Benign/ISIC_4943237_640x360.jpg  
  inflating: Resized/Benign/ISIC_4944556_640x360.jpg  
  inflating: Resized/Benign/ISIC_4948341_640x478.jpg  
  inflating: Resized/Benign/ISIC_4952820_640x480.jpg  
  inflating: Resized/Benign/ISIC_4953080_640x360.jpg  
  inflating: R

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt


IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED =42

dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "/content/Resized",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    seed=SEED
)

Found 6837 files belonging to 2 classes.


In [ ]:
dataset = dataset.map(lambda x, y: (x/255.0, y))

In [ ]:
total_batches = len(dataset)
train_batches = int(0.7 * total_batches)
val_batches   = int(0.15 * total_batches)

train_ds = dataset.take(train_batches)
val_ds   = dataset.skip(train_batches).take(val_batches)
test_ds  = dataset.skip(train_batches + val_batches)

In [ ]:
!pip install -q keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 4.9 MB/s eta 0:00:00


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

In [ ]:
def build_model(hp):
    model = keras.Sequential()

    model.add(layers.Input(shape=(224, 224, 3)))

    model.add(data_augmentation)

    # First Conv block
    model.add(layers.Conv2D(
        filters=hp.Int('filters_1', 32, 64, step=32),
        kernel_size=(3,3),
        activation='relu'
    ))
    model.add(layers.MaxPooling2D(2,2))

    # Second Conv block
    model.add(layers.Conv2D(
        filters=hp.Int('filters_2', 64, 128, step=64),
        kernel_size=(3,3),
        activation='relu'
    ))
    model.add(layers.MaxPooling2D(2,2))

    # Third Conv block
    model.add(layers.Conv2D(
        filters=hp.Int('filters_3', 128, 256, step=128),
        kernel_size=(3,3),
        activation='relu'
    ))
    model.add(layers.MaxPooling2D(2,2))

    model.add(layers.Flatten())

    # Dense layer
    model.add(layers.Dense(
        units=hp.Int('dense_units', 64, 256, step=64),
        activation='relu'
    ))

    # Dropout
    model.add(layers.Dropout(
        rate=hp.Float('dropout', 0.3, 0.6, step=0.1)
    ))

    # Output layer (binary)
    model.add(layers.Dense(1, activation='sigmoid'))

    # Learning rate tuning
    lr = hp.Choice('learning_rate', [1e-3, 1e-4])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,           # number of different models tested
    executions_per_trial=1,
    directory='cnn_tuning',
    project_name='skin_cancer'
)

In [ ]:
tuner.search(
    train_ds,
    validation_data=val_ds,
    epochs=20
)

Trial 10 Complete [00h 03m 58s]
val_accuracy: 0.91796875

Best val_accuracy So Far: 0.91796875
Total elapsed time: 01h 03m 30s


In [ ]:
!rm -rf cnn_tuning

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 22 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
test_loss, test_accuracy = best_model.evaluate(test_ds)
print("Test accuracy:", test_accuracy)

33/33 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.9018 - loss: 0.2845
Test accuracy: 0.9014354348182678


In [ ]:
from google.colab import files

best_model.save("skin_cancer_best_model.h5")
files.download("skin_cancer_best_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>